In [34]:
from ultralytics import YOLO
import os
import random
import torch
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

In [ ]:
from split_dataset import split_dataset

split_dataset("dataset", "split-dataset", train_ratio=0.8)

In [ ]:
# model trained on google colab with 100 epoch
model = YOLO("yolo26n.pt")
results = model.train(data="dataset.yaml", epochs=1, imgsz=640)

In [ ]:
trained_model_best = YOLO("content/runs/detect/train2/weights/best.pt")  # custom model trained on your dataset

In [ ]:
# Evaluate the model on a random file in the validation set

entries = os.listdir("./dataset/data")
random_filename = random.choice(entries)

results = trained_model_best(source=f"dataset/data/{random_filename}") # static image
results[0].save()  # Save the results to a directory

In [ ]:
# fine-tuned model
results_finetuned = trained_model_best.val(data="dataset.yaml")

In [ ]:
metrics = results_finetuned.box
print(f"metrics: {metrics}")

In [35]:
import onnx
import onnxruntime as ort
import cv2
import numpy as np

In [36]:
# 1. Load the session
session = ort.InferenceSession("best.onnx", providers=["CPUExecutionProvider"])

In [56]:
# 2. Preprocess the image (Manual step)
image = cv2.imread("test.jpg")
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
resized = cv2.resize(image_rgb, (640, 640))
input_tensor = resized.transpose(2, 0, 1).astype(np.float32) / 255.0
input_tensor = np.expand_dims(input_tensor, axis=0)
img_h, img_w = image.shape[:2]

# 3. Run model
outputs = session.run(None, {session.get_inputs()[0].name: input_tensor})

In [49]:
results = outputs[0]
results = results.transpose()

In [63]:
def filter_Detections(results, thresh = 0.9):
    # if model is trained on 1 class only
    if len(results[0]) == 5:
        # filter out the detections with confidence > thresh
        considerable_detections = [detection for detection in results if detection[4] > thresh]
        considerable_detections = np.array(considerable_detections)
        return considerable_detections

    # if model is trained on multiple classes
    else:
        A = []
        for detection in results:

            class_id = detection[4:].argmax()
            confidence_score = detection[4:].max()

            new_detection = np.append(detection[:4],[class_id,confidence_score])

            A.append(new_detection)

        A = np.array(A)

        # filter out the detections with confidence > thresh
        considerable_detections = [detection for detection in A if detection[-1] > thresh]
        considerable_detections = np.array(considerable_detections)

        return considerable_detections

In [64]:
results = filter_Detections(results)


In [59]:
print(results.shape)


(4, 6)


In [70]:
def NMS(boxes, conf_scores, iou_thresh = 0.9):

    #  boxes [[x1,y1, x2,y2], [x1,y1, x2,y2], ...]

    x1 = boxes[:,0]
    y1 = boxes[:,1]
    x2 = boxes[:,2]
    y2 = boxes[:,3]

    areas = (x2-x1)*(y2-y1)

    order = conf_scores.argsort()

    keep = []
    keep_confidences = []

    while len(order) > 0:
        idx = order[-1]
        A = boxes[idx]
        conf = conf_scores[idx]

        order = order[:-1]

        xx1 = np.take(x1, indices= order)
        yy1 = np.take(y1, indices= order)
        xx2 = np.take(x2, indices= order)
        yy2 = np.take(y2, indices= order)

        keep.append(A)
        keep_confidences.append(conf)

        # iou = inter/union

        xx1 = np.maximum(x1[idx], xx1)
        yy1 = np.maximum(y1[idx], yy1)
        xx2 = np.minimum(x2[idx], xx2)
        yy2 = np.minimum(y2[idx], yy2)

        w = np.maximum(xx2-xx1, 0)
        h = np.maximum(yy2-yy1, 0)

        intersection = w*h

        # union = areaA + other_areas - intesection
        other_areas = np.take(areas, indices= order)
        union = areas[idx] + other_areas - intersection

        iou = intersection/union

        boleans = iou < iou_thresh

        order = order[boleans]

        # order = [2,0,1]  boleans = [True, False, True]
        # order = [2,1]

    return keep, keep_confidences



# function to rescale bounding boxes 
def rescale_back(results,img_w,img_h):
    cx, cy, w, h, class_id, confidence = results[:,0], results[:,1], results[:,2], results[:,3], results[:,4], results[:,-1]
    cx = cx/640.0 * img_w
    cy = cy/640.0 * img_h
    w = w/640.0 * img_w
    h = h/640.0 * img_h
    x1 = cx - w/2
    y1 = cy - h/2
    x2 = cx + w/2
    y2 = cy + h/2

    boxes = np.column_stack((x1, y1, x2, y2, class_id))
    keep, keep_confidences = NMS(boxes,confidence)
    print(np.array(keep).shape)
    return keep, keep_confidences

In [71]:

rescaled_results, confidences = rescale_back(results, img_w, img_h)

(4, 5)


In [72]:
for res, conf in zip(rescaled_results, confidences):

    x1,y1,x2,y2, cls_id = res
    cls_id = int(cls_id)
    x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
    conf = "{:.2f}".format(conf)
    # draw the bounding boxes
    cv2.rectangle(image,(int(x1),int(y1)),(int(x2),int(y2)),(255,0, 0),1)


cv2.imwrite("Output.jpg", image)

True

In [46]:
from ultralytics import YOLO
yolo_model = YOLO("best.pt")

results = yolo_model("test.jpg")
results[0].save() 


image 1/1 /Users/alexjun/projects/duality/backend/apps/services/test.jpg: 640x512 2 handwritten_texts, 76.3ms
Speed: 4.5ms preprocess, 76.3ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 512)


'results_test.jpg'